# 02 · Bronze → Silver
Contexto e Objetivo

Leitura da camada Bronze para aplicação de limpeza, tipagem estrita, saneamento de dados faltantes, padronizações e enriquecimento de negócio.

In [0]:
from pyspark.sql import functions as F 
from pyspark.sql import Window

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## Limpeza dos dados

### Verificação dos tipos de dados
- Diagnóstico da Base Bruta:
- Todas as colunas vieram em formato texto puro.
- Canal em texto livre com registros inconsistentes misturando 'URA', 'ura', 'U.R.A', 'Chatbot', 'chat bot', 'Atendimento_Inicial', 'Atend. Inicial', 'ATENDIMENTO ESPECIALIZADO'.
- Motivo em texto livre com variações de caixa e acentuação ('CONSULTA DE FATURA', 'Bloqueio de cartao').
- Resolvido com registros inconsistentes misturando 'Sim', 'S', '1', 'Não', 'N', '0', 'Nao resolvido' e texto vazio ("").
- Tempo_espera e tempo_atendimento com o texto 'NULL' no lugar de valores ausentes.
- 2.636 registros duplicados.

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")

display(df_bronze_chamados.limit(5))

In [0]:
for c in df_bronze_chamados.columns:
    display(
        df_bronze_chamados.select(
            F.count("*").alias(c),
            F.sum(F.col(c).isNull().cast("int")).alias("nulos"),
            F.sum((F.trim(F.col(c)) == "").cast("int")).alias("vazios")
        )
    )

    display(
        df_bronze_chamados.groupBy(c).count()
                          .orderBy(F.desc("count"))
                          .limit(50)
    )

### Tratamento de valores em 'canais', 'motivos' e 'resolvido'

#### Padronização de Canais:

- 'ura', 'u r a' $\rightarrow$ URA
- 'chatbot', 'chat bot' $\rightarrow$ Chatbot
- 'atendimento inicial', 'atend inicial' $\rightarrow$ Atendimento Inicial
- 'atendimento especializado', 'atend especializado' $\rightarrow$ Atendimento Especializado

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")
df_silver_chamados = (
    df_bronze_chamados
        .withColumn("canal", F.lower(F.col("canal")))
        .withColumn("canal", F.regexp_replace(F.col("canal"), r"[^a-z]+", " "))
        .withColumn("canal", F.trim(F.col("canal")))
        .withColumn("canal",
            F.when(F.col("canal").isin("ura", "u r a"), "URA")
            .when(F.col("canal").isin("chatbot", "chat bot"), "Chatbot")
            .when(F.col("canal").isin("atendimento inicial", "atend inicial"), "Atendimento Inicial")
            .when(F.col("canal").isin("atendimento especializado", "atend especializado"), "Atendimento Especializado")
            .otherwise("Não Informado")
    )
)

display(df_silver_chamados.groupBy("canal").count().orderBy(F.desc("count")))


#### Padronização de Motivos: 
lower, remoção de acentos (unicodedata), remoção de caracteres especiais e trim, harmonizando para os 14 motivos oficiais do case (ex.: 'bloqueio de cartao' $\rightarrow$ Bloqueio de cartão).

In [0]:
import unicodedata
from pyspark.sql.types import StringType

@F.udf(StringType())
def remover_acentos(texto):
    if texto is None:
        return None
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

df_silver_chamados = (
    df_silver_chamados
        .withColumn("motivo", F.lower(F.col("motivo")))
        .withColumn("motivo", remover_acentos(F.col("motivo")))
        .withColumn("motivo", F.regexp_replace(F.col("motivo"), r"[^a-z]+", " "))
        .withColumn("motivo", F.trim(F.col("motivo")))
        .withColumn("motivo",
            F.when(F.col("motivo") == "consulta de fatura", "Consulta de fatura")
            .when(F.col("motivo") == "consulta de limite", "Consulta de limite")
            .when(F.col("motivo") == "consulta de contrato", "Consulta de contrato")
            .when(F.col("motivo") == "contestacao de fatura", "Contestação de fatura")
            .when(F.col("motivo") == "renegociacao de divida", "Renegociação de dívida")
            .when(F.col("motivo") == "bloqueio de cartao", "Bloqueio de cartão")
            .when(F.col("motivo") == "desbloqueio de cartao", "Desbloqueio de cartão")
            .when(F.col("motivo") == "contratacao de cartao adicional", "Contratação de cartão adicional")
            .when(F.col("motivo") == "compra nao autorizada", "Compra não autorizada")
            .when(F.col("motivo") == "alteracao de dados cadastrais vencimento da fatura telefone email", "Alteração de dados cadastrais")
            .when(F.col("motivo") == "transferencia de agencia", "Transferência de agência")
            .when(F.col("motivo") == "cancelamento de conta", "Cancelamento de conta")
            .when(F.col("motivo") == "problema com aplicativo", "Problema com aplicativo")
            .when(F.col("motivo") == "duvidas gerais sobre programa de pontos", "Dúvidas gerais sobre programa de pontos")
            .otherwise("Não Informado")
        )
)

display(df_silver_chamados.groupBy("motivo").count().orderBy(F.desc("count")))


#### Padronização de Resolvido:
- 'sim', 's', '1' $\rightarrow$ Sim
- 'não', 'n', '0', 'nao resolvido' $\rightarrow$ Não
- Vazio/nulo $\rightarrow$ Não Informado

In [0]:
df_silver_chamados = (
    df_silver_chamados
        .withColumn("resolvido", F.lower(F.col("resolvido")))
        .withColumn("resolvido", F.trim(F.col("resolvido")))
        .withColumn("resolvido",
            F.when(F.col("resolvido").isin("não", "n", "0", "nao resolvido"), "Não")
            .when(F.col("resolvido").isin("sim", "s", "1"), "Sim")
            .otherwise("Não Informado")
    )
)

display(df_silver_chamados.groupBy("resolvido").count().orderBy(F.desc("count")))


### Tipagem:
tempo_espera, tempo_atendimento e id_atendente convertidos para IntegerType() com try_cast.

In [0]:
df_silver_chamados = (
    df_silver_chamados
        .withColumn("tempo_espera", F.expr("try_cast(tempo_espera as int)"))
        .withColumn("tempo_atendimento", F.expr("try_cast(tempo_atendimento as int)"))
        .withColumn("id_atendente", F.expr("try_cast(id_atendente as int)"))
)

df_silver_chamados.printSchema()

### Linhas duplicadas

#### Verificar dados duplicados

In [0]:
print("linhas:", df_bronze_chamados.count())
print("ids distintos:", df_bronze_chamados.select("id_chamado", "id_cliente", "canal", "motivo", "resolvido").distinct().count())
print("ids duplicados:", df_bronze_chamados.count() - df_bronze_chamados.select("id_chamado", "id_cliente", "canal", "motivo", "resolvido").distinct().count())

#### Remover valores duplicados

dropDuplicates por id_chamado, id_cliente, canal, motivo e resolvido.

In [0]:
print("linhas antes:", df_silver_chamados.count())

df_silver_chamados = (
    df_silver_chamados
    .dropDuplicates(["id_chamado", "id_cliente", "canal", "motivo", "resolvido"])
)

print("linhas depois:", df_silver_chamados.count())

## Imputação dos 5 Eixos Oficiais do Case (categoria_motivos):
- Consulta de fatura, Contestação de fatura, Consulta de limite, Consulta de contrato, Renegociação de dívida $\rightarrow$ Financeiro
- Bloqueio de cartão, Desbloqueio de cartão, Contratação de cartão adicional, Compra não autorizada $\rightarrow$ Cartão
- Alteração de dados cadastrais, Transferência de agência, Cancelamento de conta $\rightarrow$ Cadastral
- Problema com aplicativo $\rightarrow$ Atendimento
- Dúvidas gerais sobre programa de pontos $\rightarrow$ Benefícios

In [0]:
#criar coluna categoria_motivos
df_silver_chamados = (
    df_silver_chamados
    .withColumn("categoria_motivos",
        F.when(F.col("motivo").isin("Consulta de fatura", 
                                   "Contestação de fatura",
                                   "Consulta de limite", 
                                   "Consulta de contrato", 
                                   "Renegociação de dívida"), "Financeiro")
            .when(F.col("motivo").isin("Bloqueio de cartão",
                                      "Desbloqueio de cartão",
                                      "Contratacao de cartão adicional",
                                      "Compra não autorizada"), "Cartão")
            .when(F.col("motivo").isin("Alteração de dados cadastrais",
                                      "Transferência de agência",
                                      "Cancelamento de conta"), "Cadastral")
            .when(F.col("motivo").isin("Problema com aplicativo"), "Atendimento")
            .when(F.col("motivo").isin("Dúvidas gerais sobre programa de pontos"), "Benefícios")
    )
)

display(df_silver_chamados.limit(20))


## Gravação do tb_chamados

In [0]:
df_silver_chamados.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_chamados")

display(df_bronze_chamados.limit(10))
display(df_silver_chamados.limit(10))
